# M3 Architecture Sweeps

Slurm-only control notebook for running non-DeepConvLSTM M3 architecture experiments without mixing outputs with the completed DeepConvLSTM `full_eXX` folders. The wrapper writes to `arch_sweeps/<model_variant>/<experiment_code>` by default.

In [ ]:
%%bash
set -euo pipefail
cd /shared/b00088568/github/har-mcu
echo "Registered Conv2D-safe variants:"
printf '%s\n' \
  daghero_cnn_2layer_conv2d \
  repmobile_folded_conv2d \
  tcn_attention_har_teacher_conv2d \
  tcn_inception_conv2d \
  xtinyhar_student_conv2d
echo
echo "Default output pattern: reports/m3/arch_sweeps/<model_variant>/<experiment_code>/"
echo "Default TFLite pattern: models_tflite/m3/<experiment_id>/arch_sweeps/<model_variant>/<experiment_code>/"


In [ ]:
%%bash
set -euo pipefail
cd /shared/b00088568/github/har-mcu
CONFIG=configs/m3/E00_wisdm_m2_anchor.yaml
VARIANTS=(
  daghero_cnn_2layer_conv2d
  repmobile_folded_conv2d
  tcn_attention_har_teacher_conv2d
  tcn_inception_conv2d
  xtinyhar_student_conv2d
)
for VARIANT in "${VARIANTS[@]}"; do
  CMD="bash scripts/slurm/submit_m3_arch_experiment.sh ${CONFIG} ${VARIANT} --smoke --max-windows-per-class 20 --representative-samples 16"
  echo "$CMD"
  if [ "${SUBMIT_ARCH_SMOKES:-0}" = "1" ]; then eval "$CMD"; else echo "Set SUBMIT_ARCH_SMOKES=1 to submit."; fi
done


In [ ]:
%%bash
set -euo pipefail
cd /shared/b00088568/github/har-mcu
MODEL_VARIANT=${MODEL_VARIANT:-daghero_cnn_2layer_conv2d}
CONFIGS=(
  configs/m3/E00_wisdm_m2_anchor.yaml
  configs/m3/E03_arduino_downsample_20hz_T100.yaml
  configs/m3/E04_wisdm_to_g_arduino_g.yaml
  configs/m3/E05_legacy_arduino_to_mps2.yaml
  configs/m3/E06_no_norm_matched.yaml
  configs/m3/E07_skip_inference_norm_diag.yaml
  configs/m3/E08_T50_window.yaml
  configs/m3/E09_wisdm_pretrain_arduino_finetune.yaml
  configs/m3/E10_arduino_from_scratch.yaml
)
for CONFIG in "${CONFIGS[@]}"; do
  CMD="bash scripts/slurm/submit_m3_arch_experiment.sh ${CONFIG} ${MODEL_VARIANT}"
  echo "$CMD"
  if [ "${SUBMIT_ARCH_FULL:-0}" = "1" ]; then eval "$CMD"; else echo "Set SUBMIT_ARCH_FULL=1 to submit."; fi
done


In [ ]:
%%bash
set -euo pipefail
cd /shared/b00088568/github/har-mcu
squeue -u "$USER"
if [ -n "${JOBID:-}" ]; then
  sacct -j "$JOBID" --format=JobID,JobName%45,State,ExitCode,Elapsed,Timelimit,Start,End -P
  find slurm_logs -type f | grep "\\.${JOBID}\\." || true
fi
